# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|█████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:58<00:00, 39.59s/it]


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

'Title: SwissGear 1900 Mini/Slim TSA-Friendly 13" Backpack for $46 + free shipping\nDetails: Clip the coupon on the product page to get this deal. It\'s $39 off and the lowest price we could find. Buy Now at Amazon\nFeatures: Fits most 13" laptops TSA-friendly lay-flat design Rugged polyester material Ergonomic, padded shoulder straps Includes RFID-protected organizer\nURL: https://www.dealnews.com/Swiss-Gear-1900-Mini-Slim-TSA-Friendly-13-Backpack-for-46-free-shipping/21812111.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [8]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [9]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Energizer Deals at Amazon: Up to 51% off + extra 30% off via Sub & Save
Details: Save on batteries, flashlights, headlamps, and more. Some get extra discounts with Subscribe and Save. We've pictured the Energizer Ultimate Lithium AAA Batteries 24 Count for $27 after Subscribe and Save ($39 off). Shop Now at Amazon
Features: 
URL: https://www.dealnews.com/Energizer-Deals-at-Amaz

In [10]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Energizer Ultimate Lithium AAA Batteries, 24-count. These are high-performance lithium primary batteries designed for longer shelf life and reliable power in high-drain devices like digital cameras, flashlights, and headlamps. The pack provides consistent voltage under heavy load and performs well in extreme temperatures, making it suitable for photography, outdoor gear, and emergency kits.', price=27.0, url='https://www.dealnews.com/Energizer-Deals-at-Amazon-Up-to-51-off-extra-30-off-via-Sub-Save/21812121.html?iref=rss-c142'), Deal(product_description='Unlocked Apple iPhone 14 Pro, 128GB (refurbished). This is a refurbished unlocked iPhone 14 Pro with 128GB of storage, offering the Pro-series camera system, powerful A-series performance, and OLED display technology in a compact flagship form. The listing includes a 1-year Allstate warranty, making it a lower-cost option for users seeking near-original hardware and software capabilities wi

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Energizer Ultimate Lithium AAA Batteries, 24-count. These are high-performance lithium primary batteries designed for longer shelf life and reliable power in high-drain devices like digital cameras, flashlights, and headlamps. The pack provides consistent voltage under heavy load and performs well in extreme temperatures, making it suitable for photography, outdoor gear, and emergency kits.
27.0
https://www.dealnews.com/Energizer-Deals-at-Amazon-Up-to-51-off-extra-30-off-via-Sub-Save/21812121.html?iref=rss-c142

Unlocked Apple iPhone 14 Pro, 128GB (refurbished). This is a refurbished unlocked iPhone 14 Pro with 128GB of storage, offering the Pro-series camera system, powerful A-series performance, and OLED display technology in a compact flagship form. The listing includes a 1-year Allstate warranty, making it a lower-cost option for users seeking near-original hardware and software capabilities without carrier restrictions.
345.0
https://www.dealnews.com/products/Apple/Unlocked-Apple-

In [12]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [18]:
result

DealSelection(deals=[Deal(product_description='Energizer Ultimate Lithium AAA Batteries, 24-count pack. These are high-energy, long-lasting AAA lithium cells designed for devices that demand reliable power and extended shelf life, such as digital cameras, flashlights, and high-drain electronics. Lithium chemistry offers lighter weight, superior cold-weather performance, and longer runtime compared with alkaline batteries.', price=27.0, url='https://www.dealnews.com/Energizer-Deals-at-Amazon-Up-to-51-off-extra-30-off-via-Sub-Save/21812121.html?iref=rss-c142'), Deal(product_description="Unlocked Apple iPhone 14 Pro, 128GB refurbished. A flagship-class smartphone featuring Apple's Pro camera system, high-performance chipset, and a 128GB internal storage capacity; sold refurbished and backed by a one-year Allstate warranty. It provides Apple's iOS experience in an unlocked form factor suitable for use on multiple carriers.", price=345.0, url='https://www.dealnews.com/products/Apple/Unlocke

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [16]:
load_dotenv(override=True)

True

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")